# 🔍 Fake Internship Detection Using Machine Learning

> **Data Science Mini Project** | NLP + Machine Learning

This notebook detects whether an internship/job posting is **Fake** or **Legitimate** using:
- **NLP** (NLTK stopword removal, text cleaning)
- **TF-IDF Vectorization**
- **Multiple ML Classifiers** (Logistic Regression, Naive Bayes, Random Forest, SVM)

---
**Dataset:** [Kaggle – Fake Job Postings](https://www.kaggle.com/datasets/shivamb/real-or-fake-fake-jobpostings)  
Place `fake_job_postings.csv` in the `dataset/` folder before running.

## CELL 1 — Install & Import Libraries

In [ ]:
# Uncomment below if running in Google Colab
# !pip install pandas numpy scikit-learn matplotlib seaborn nltk wordcloud joblib -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import string
import warnings
import joblib

import nltk
nltk.download('stopwords')
nltk.download('punkt')
from nltk.corpus import stopwords

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, classification_report)

warnings.filterwarnings('ignore')
print('✅ All libraries imported successfully!')

## CELL 2 — Load the Dataset

In [ ]:
# === FOR GOOGLE COLAB: Upload dataset manually ===
# from google.colab import files
# uploaded = files.upload()
# df = pd.read_csv('fake_job_postings.csv')

# === FOR JUPYTER NOTEBOOK (standard) ===
df = pd.read_csv('../dataset/fake_job_postings.csv')

print(f'✅ Dataset loaded: {df.shape[0]} rows, {df.shape[1]} columns')
df.head()

## CELL 3 — Explore the Dataset

In [ ]:
print('Column Info:')
print(df.info())

print('\nTarget Variable Distribution:')
print(df['fraudulent'].value_counts())

print('\nMissing Values:')
print(df.isnull().sum())

## CELL 4 — Visualize: Fake vs Legitimate Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Fake vs Legitimate Job Postings', fontsize=16, fontweight='bold')

counts = df['fraudulent'].value_counts()
labels = ['Legitimate (0)', 'Fake (1)']
colors = ['#2ecc71', '#e74c3c']

# Bar chart
axes[0].bar(labels, counts.values, color=colors, edgecolor='black', width=0.5)
axes[0].set_title('Count of Job Postings')
axes[0].set_ylabel('Count')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 50, str(v), ha='center', fontweight='bold')

# Pie chart
axes[1].pie(counts.values, labels=labels, colors=colors,
            autopct='%1.1f%%', startangle=140)
axes[1].set_title('Proportion')

plt.tight_layout()
plt.savefig('../models/distribution_plot.png', dpi=150, bbox_inches='tight')
plt.show()

## CELL 5 — Data Preprocessing

In [ ]:
# Select important columns
useful_columns = ['title', 'company_profile', 'description',
                  'requirements', 'location', 'salary_range', 'fraudulent']
df = df[useful_columns]

# Fill missing text with empty string
text_cols = ['title', 'company_profile', 'description',
             'requirements', 'location', 'salary_range']
df[text_cols] = df[text_cols].fillna('')

# Remove duplicates
before = len(df)
df = df.drop_duplicates()
print(f'Removed {before - len(df)} duplicates. Rows remaining: {len(df)}')

# Ensure target is integer
df['fraudulent'] = df['fraudulent'].astype(int)
print(f'Dataset shape: {df.shape}')

## CELL 6 — Feature Engineering: Combine Text Columns

In [ ]:
# Combine all text columns into one 'text_data' column
df['text_data'] = (df['title'] + ' ' + df['company_profile'] + ' ' +
                   df['description'] + ' ' + df['requirements'] + ' ' +
                   df['location'] + ' ' + df['salary_range'])

print('✅ Created text_data column')
print(f'Sample: {df["text_data"][0][:200]}...')

## CELL 7 — NLP Text Cleaning Function

In [ ]:
stop_words = set(stopwords.words('english'))

def clean_text(text):
    """Clean raw text: lowercase, remove URLs/HTML/punctuation/numbers/stopwords."""
    text = text.lower()
    text = re.sub(r'http\S+|www\.\S+', '', text)       # Remove URLs
    text = re.sub(r'<.*?>', '', text)                   # Remove HTML
    text = text.translate(str.maketrans('', '', string.punctuation))  # Remove punctuation
    text = re.sub(r'\d+', '', text)                     # Remove digits
    tokens = [w for w in text.split() if w not in stop_words and len(w) > 2]
    return ' '.join(tokens)

print('⏳ Cleaning text... please wait...')
df['clean_text'] = df['text_data'].apply(clean_text)
print('✅ Text cleaning done!')
print(f'\nCleaned sample: {df["clean_text"][0][:200]}')

## CELL 8 — TF-IDF Vectorization

In [ ]:
# TF-IDF: Convert text to a numerical matrix
tfidf = TfidfVectorizer(
    max_features=10000,   # Use top 10,000 words
    ngram_range=(1, 2),   # Unigrams and bigrams
    sublinear_tf=True     # Log normalization
)

X = tfidf.fit_transform(df['clean_text'])   # Feature matrix
y = df['fraudulent']                         # Target labels

print(f'✅ TF-IDF complete!')
print(f'   Feature matrix: {X.shape}')

## CELL 9 — Train-Test Split

In [ ]:
# 80% training, 20% testing
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f'Training samples: {X_train.shape[0]}')
print(f'Testing  samples: {X_test.shape[0]}')

## CELL 10 — Train Multiple ML Models

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, C=1.0, random_state=42),
    'Naive Bayes':         MultinomialNB(alpha=0.1),
    'Random Forest':       RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    'SVM (LinearSVC)':     LinearSVC(C=1.0, max_iter=2000, random_state=42),
}

results = {}

for model_name, model in models.items():
    print(f'⏳ Training {model_name}...')
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    results[model_name] = {
        'model':     model,
        'y_pred':    y_pred,
        'accuracy':  accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred, zero_division=0),
        'recall':    recall_score(y_test, y_pred, zero_division=0),
        'f1_score':  f1_score(y_test, y_pred, zero_division=0),
    }
    print(f'   Accuracy: {results[model_name]["accuracy"]*100:.2f}%  |  F1: {results[model_name]["f1_score"]*100:.2f}%')

## CELL 11 — Classification Reports

In [ ]:
for model_name, res in results.items():
    print(f'\n{"="*50}')
    print(f'Model: {model_name}')
    print('='*50)
    print(classification_report(y_test, res['y_pred'],
                                target_names=['Legitimate', 'Fake']))

## CELL 12 — Model Performance Comparison (Chart)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Model Performance Comparison', fontsize=16, fontweight='bold')

model_names = list(results.keys())
metrics = {
    'Accuracy':  [results[m]['accuracy']  for m in model_names],
    'Precision': [results[m]['precision'] for m in model_names],
    'Recall':    [results[m]['recall']    for m in model_names],
    'F1 Score':  [results[m]['f1_score']  for m in model_names],
}

x = np.arange(len(model_names))
width = 0.2
bar_colors = ['#3498db', '#2ecc71', '#e74c3c', '#f39c12']

for i, (metric, values) in enumerate(metrics.items()):
    axes[0].bar(x + i * width, values, width, label=metric, color=bar_colors[i], alpha=0.85)

axes[0].set_title('All Metrics by Model')
axes[0].set_xticks(x + width * 1.5)
axes[0].set_xticklabels(model_names, rotation=15, ha='right', fontsize=9)
axes[0].set_ylim(0.8, 1.02)
axes[0].legend(fontsize=9)
axes[0].grid(axis='y', alpha=0.3)

accuracies = [results[m]['accuracy']*100 for m in model_names]
barh = axes[1].barh(model_names, accuracies,
                    color=['#8e44ad','#16a085','#c0392b','#d35400'],
                    edgecolor='black', height=0.5)
axes[1].set_title('Accuracy Comparison (%)')
axes[1].set_xlim(85, 100)
for bar, val in zip(barh, accuracies):
    axes[1].text(val + 0.1, bar.get_y() + bar.get_height()/2,
                f'{val:.2f}%', va='center', fontweight='bold')

plt.tight_layout()
plt.savefig('../models/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## CELL 13 — Confusion Matrices

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Confusion Matrices for All Models', fontsize=16, fontweight='bold')
axes = axes.flatten()

for idx, (model_name, res) in enumerate(results.items()):
    cm = confusion_matrix(y_test, res['y_pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx],
                xticklabels=['Predicted Real','Predicted Fake'],
                yticklabels=['Actual Real','Actual Fake'])
    axes[idx].set_title(f'{model_name}  (Acc: {res["accuracy"]*100:.1f}%)')

plt.tight_layout()
plt.savefig('../models/confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

## CELL 14 — Save Best Model to Disk

In [ ]:
# Identify best model by F1 Score
best_model_name = max(results, key=lambda m: results[m]['f1_score'])
best_model      = results[best_model_name]['model']

print(f'🏆 Best Model: {best_model_name}')
print(f'   F1 Score: {results[best_model_name]["f1_score"]*100:.2f}%')

# Save to disk
joblib.dump(best_model, '../models/trained_model.pkl')
joblib.dump(tfidf,      '../models/tfidf_vectorizer.pkl')

print('✅ Model saved: models/trained_model.pkl')
print('✅ Vectorizer saved: models/tfidf_vectorizer.pkl')

## CELL 15 — Prediction System

In [ ]:
def predict_internship(description, model=best_model, vectorizer=tfidf):
    """
    Predict whether a job description is Fake or Legitimate.

    Args:
        description (str): Raw internship/job description
    Returns:
        str: Prediction label
    """
    cleaned    = clean_text(description)
    vectorized = vectorizer.transform([cleaned])
    prediction = model.predict(vectorized)[0]
    return '🚨 FAKE Internship — DO NOT Apply!' if prediction == 1 else '✅ Legitimate Internship — Looks Safe!'


# --- Test Cases ---
test_cases = [
    ('🔴 FAKE Test',  'Work from home. Pay Rs.500 registration fee. Earn Rs.50,000 guaranteed.'),
    ('🔴 FAKE Test',  'Urgently hiring! No skills required. Send bank details now. Earn $5000/week.'),
    ('🟢 LEGIT Test', 'Software Engineering Intern at Google. Requires Python skills. 3-month paid internship. Stipend: ₹20,000/month.'),
    ('🟢 LEGIT Test', 'Marketing Intern. BBA/MBA candidate. Social media management. 6-month paid internship at ABC Pvt. Ltd.'),
]

print('🔍 Testing Prediction System:\n')
for label, text in test_cases:
    result = predict_internship(text)
    print(f'{label}: {text[:70]}...')
    print(f'Result → {result}\n')

## CELL 16 — Interactive Prediction

In [ ]:
# === INTERACTIVE MODE ===
# Uncomment to enter your own description

# user_input = input('Paste the internship description:\n> ')
# print(predict_internship(user_input))

# === QUICK TEST ===
my_description = "Work from home. No experience needed. Pay a small registration fee of Rs.999. Get paid Rs.30,000 monthly."
print(predict_internship(my_description))